In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import warnings
warnings.filterwarnings("ignore")
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

print(f"Using device: {device}")

In [2]:
@dataclass
class BridgeConfig:
    vision_dim: int = 1024  # ViT-L/14 output dimension
    llm_dim: int = 4096  # LLaMA-7B hidden dimension
    num_patches: int = 256  # 16×16 patch grid from 224×224 image
    num_queries: int = 64  # Flamingo / BLIP-2 learned query count
    batch_num: int = 4  # Mini-batch size for demos
    num_heads: int = 8  # Multi-head attention heads
    num_levels: int = 4  # ViT intermediate layers for DeepStack


cfg = BridgeConfig()


def make_visual_features(
    batch_num: int, num_patches: int, vision_dim: int
) -> torch.Tensor:
    """Simulate frozen ViT output: CLS token prepended to patch tokens."""
    # (batch_num, num_patches + 1, vision_dim)
    return torch.randn(batch_num, num_patches + 1, vision_dim, device=device)


def make_multilevel_features(
    batch_num: int, num_patches: int, vision_dim: int, num_levels: int
) -> List[torch.Tensor]:
    """Simulate outputs from multiple intermediate ViT transformer blocks."""
    # List of num_levels tensors, each: (batch_num, num_patches, vision_dim)
    return [
        torch.randn(batch_num, num_patches, vision_dim, device=device)
        for _ in range(num_levels)
    ]

# 1) MLP Projector

> **Paper:** LLaVA 1.5 — *"Improved Baselines with Visual Instruction Tuning"* (Liu et al. 2023)  
> **Used by:** LLaVA 1.5, Qwen-VL, InternVL, LLaVA-Next, Bunny, …

- LLaVA 1.0 used a **single linear layer** to bridge vision → language.  
- LLaVA 1.5's key finding: adding **one more linear + GELU** dramatically improves performance.  
- Adding a third layer brings no measurable gain.

<img src="https://miro.medium.com/v2/resize:fit:1400/1*k_qqcTTyxtTgC1Q1o9akEg.png" width="700">

**No compression**: all $N$ patch tokens pass through unchanged.  
The LLM sequence length is `(num_patches + num_text_tokens)`.

In [3]:
class MLPProjector(nn.Module):
    """
    Two-layer MLP projector — the de facto standard since LLaVA 1.5 (2023).

    Architecture:
        Linear(vision_dim → llm_dim) → GELU → Linear(llm_dim → llm_dim)

    Design decisions:
    - Hidden dim == output dim (not a bottleneck) because we want rich capacity
      for the nonlinear remap, not compression.
    - GELU matches the activation in most LLM FFN blocks, keeping gradient
      statistics consistent at the LLM input boundary.
    - Xavier uniform init avoids early saturation when vision_dim ≠ llm_dim.
    """

    def __init__(self, vision_dim: int, llm_dim: int) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )
        self._init_weights()

    def _init_weights(self) -> None:
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, visual_features: torch.Tensor) -> torch.Tensor:
        # Input: (batch_num, num_patches, vision_dim)
        # Output: (batch_num, num_patches, llm_dim)
        return self.mlp(visual_features)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


# Demo
visual_feats = make_visual_features(cfg.batch_num, cfg.num_patches, cfg.vision_dim)
mlp_proj = MLPProjector(cfg.vision_dim, cfg.llm_dim).to(device)
mlp_out = mlp_proj(visual_feats)

print("Input: ", visual_feats.size())
print("Output: ", mlp_out.size())

Input:  torch.Size([4, 257, 1024])
Output:  torch.Size([4, 257, 4096])


# 2) Cross-Attention Resampler  *(Flamingo)*

> **Paper:** Flamingo — *"A Visual Language Model for Few-Shot Learning"* (Alayrac et al. 2022, DeepMind)  
> **Also used by:** OpenFlamingo, IDEFICS, Otter

Flamingo's insight: you don't need to hand *all* $N$ patch tokens to the LLM.  
Instead, learn $K$ **query vectors** that selectively *pull out* the most relevant information via cross-attention.

<img src="https://miro.medium.com/v2/resize:fit:1400/0*Jrv0c8N2-ol2hx6f" width=500>

Think of each query as a "question" asked of the image:
- Query 1: *"What objects are in the foreground?"*
- Query 2: *"What is the background scene?"*
- Query 3: *"Are there any text or numbers?"*
- … (the network learns what questions to ask)

This **compresses** $N=256$ patch tokens down to $K=64$ query tokens — a 4× reduction in LLM sequence length, saving $O(K^2)$ attention compute inside the LLM.

In [4]:
class PerceiverLayer(nn.Module):
    """
    Single Perceiver block: cross-attention (queries attend to visual K/V) + FFN.
    Pre-norm (LayerNorm before attention) for training stability.
    """

    def __init__(self, llm_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            llm_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(llm_dim, llm_dim * 4),
            nn.GELU(),
            nn.Linear(llm_dim * 4, llm_dim),
        )
        self.norm1 = nn.LayerNorm(llm_dim)
        self.norm2 = nn.LayerNorm(llm_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries: torch.Tensor, kv: torch.Tensor) -> torch.Tensor:
        # Pre-norm cross-attention: queries (text) are Q, visual features are K and V
        # queries: (batch_num, num_queries, llm_dim)
        # kv:      (batch_num, num_patches, llm_dim)

        residual = queries

        # Each token in num_queries attend to all num_patches
        attn_out, _ = self.cross_attn(query=self.norm1(queries), key=kv, value=kv)
        queries = residual + self.dropout(attn_out)

        # FFN with residual connection
        # (batch_num, num_queries, llm_dim) → (batch_num, num_queries, llm_dim)
        residual = queries
        queries = residual + self.dropout(self.ffn(self.norm2(queries)))
        return queries


class PerceiverResampler(nn.Module):
    """
    Cross-attention resampler from Flamingo (Alayrac et al. 2022).

    Key parameters:
        num_queries: K — the number of output tokens. Flamingo uses 64.
        num_layers:  depth of Perceiver stack. More layers = richer compression.

    The learned query vectors (self.queries) are the only persistent visual
    representation — they are *not* image-specific; they are task-level slots
    that get filled in differently per image during the forward pass.
    """

    def __init__(
        self,
        vision_dim: int,
        llm_dim: int,
        num_queries: int = 64,
        num_heads: int = 8,
        num_layers: int = 2,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_queries = num_queries

        # Persistent learned query slots — shared across the batch
        # (num_queries, llm_dim)
        self.queries = nn.Parameter(torch.randn(num_queries, llm_dim) * 0.02)

        # Project visual features from vision_dim → llm_dim (key/value space)
        self.kv_proj = nn.Linear(vision_dim, llm_dim)

        # Stack of Perceiver layers
        self.layers = nn.ModuleList(
            [PerceiverLayer(llm_dim, num_heads, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(llm_dim)

    def forward(self, visual_features: torch.Tensor) -> torch.Tensor:
        batch_num = visual_features.shape[0]

        # Project visual keys/values to llm_dim
        # (batch_num, num_patches, vision_dim) → (batch_num, num_patches, llm_dim)
        kv = self.kv_proj(visual_features)

        # Broadcast learned queries over the batch dimension
        # (num_queries, llm_dim) → (batch_num, num_queries, llm_dim)
        queries = self.queries.unsqueeze(0).expand(batch_num, -1, -1)

        for layer in self.layers:
            # (batch_num, num_queries, llm_dim)
            queries = layer(queries, kv)

        # (batch_num, num_queries, llm_dim)
        return self.norm(queries)

# 3) Q-Former  *(BLIP-2)*

> **Paper:** BLIP-2 — *"Bootstrapping Language-Image Pre-training"* (Li et al. 2023, Salesforce)  
> **Status:** Historically important; largely superseded by MLP projectors in practice.

<img src="https://miro.medium.com/v2/resize:fit:1400/0*rQPMuhmyjCnh7OKX.png" width=500>

Q-Former goes further than the Perceiver by adding **self-attention among the queries** (not just cross-attention to visual features). This allows queries to communicate with each other and de-duplicate redundant information.

The architectural novelty of BLIP-2 was not the Q-Former itself, but its **three-objective pre-training**:

| Objective | What it learns |
|-----------|----------------|
| Image-Text Contrastive (ITC) | Align query representations with text embeddings |
| Image-grounded Text Matching (ITM) | Binary: does the text match the image? |
| Image-grounded Text Generation (ITG) | Generate causal text conditioned on queries |

This pre-training made Q-Former a *standalone transferable bridge* — you could freeze both ViT and LLM and only train Q-Former on image-text pairs.

**Why It Was Superseded**

- Adds ~200M parameters with a separate pre-training stage
- MLP projector (~20M params) trained end-to-end on instruction data matches or beats it
- Complex 3-objective training is hard to reproduce and tune

In [5]:
class QFormerLayer(nn.Module):
    """
    Single Q-Former transformer layer.

    Difference from standard transformer layer:
      Standard: self-attn(all tokens) → cross-attn(all tokens, context) → FFN
      Q-Former:  self-attn(queries only)  → cross-attn(queries, visual) → FFN

    The queries never mix with text tokens at the layer level — only via the
    pre-training objectives (ITC/ITM/ITG) which are not implemented here.
    """

    def __init__(self, model_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            model_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            model_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.GELU(),
            nn.Linear(model_dim * 4, model_dim),
        )
        self.norm1 = nn.LayerNorm(model_dim)
        self.norm2 = nn.LayerNorm(model_dim)
        self.norm3 = nn.LayerNorm(model_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        queries: torch.Tensor,
        visual_kv: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Self-attention: queries attend to each other (de-duplication)

        # Input (batch_num, num_queries, model_dim)
        # Output (batch_num, num_queries, model_dim)
        residual = queries
        sa_out, _ = self.self_attn(
            self.norm1(queries), self.norm1(queries), self.norm1(queries)
        )
        queries = residual + self.dropout(sa_out)

        # Cross-attention: queries extract information from visual features
        # queries: (batch_num, num_queries, model_dim) — Q
        # visual_kv: (batch_num, num_patches, model_dim) — K and V
        residual = queries
        ca_out, attn_w = self.cross_attn(
            query=self.norm2(queries), key=visual_kv, value=visual_kv
        )
        queries = residual + self.dropout(ca_out)

        # FFN
        # Input: (batch_num, num_queries, model_dim)
        # Output: (batch_num, num_queries, model_dim)
        residual = queries
        queries = residual + self.dropout(self.ffn(self.norm3(queries)))

        return queries, attn_w


class QFormer(nn.Module):
    """
    Querying Transformer from BLIP-2 (Li et al. 2023).

    Simplified version: we implement the architecture but not the 3-objective
    pre-training (ITC / ITM / ITG), as that requires paired image-text data
    and is tangential to understanding the architectural contribution.

    Key parameters (matching BLIP-2 paper):
        num_queries = 32   (BLIP-2 uses 32)
        num_layers  = 6    (BLIP-2 uses a BERT_base-scale Q-Former: 12 layers)
        model_dim   = 768  (reduced here for demo)
    """

    def __init__(
        self,
        vision_dim: int,
        model_dim: int,
        llm_dim: int,
        num_queries: int = 32,
        num_heads: int = 8,
        num_layers: int = 6,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.num_queries = num_queries

        # Learned query tokens — initialised like BERT [CLS]: truncated normal
        # (num_queries, model_dim) — shared, not image-specific
        self.query_tokens = nn.Parameter(torch.zeros(num_queries, model_dim))
        nn.init.trunc_normal_(self.query_tokens, std=0.02)

        # Project ViT features to Q-Former's internal space
        self.visual_proj = nn.Linear(vision_dim, model_dim)

        # Q-Former transformer blocks
        self.layers = nn.ModuleList(
            [QFormerLayer(model_dim, num_heads, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(model_dim)

        # Final projection: Q-Former dim → LLM dim
        self.output_proj = nn.Linear(model_dim, llm_dim)

    def forward(
        self, visual_features: torch.Tensor
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        batch_num = visual_features.shape[0]

        # Project visual features to Q-Former's model_dim (K and V space)
        # Input: (batch_num, num_patches, vision_dim)
        # Output: (batch_num, num_patches, model_dim)
        visual_kv = self.visual_proj(visual_features)

        # Broadcast learnable query tokens over batch
        # (num_queries, model_dim) → (batch_num, num_queries, model_dim)
        queries = self.query_tokens.unsqueeze(0).expand(batch_num, -1, -1)

        all_attn_weights: List[torch.Tensor] = []
        for layer in self.layers:
            # (batch_num, num_queries, model_dim)
            queries, attn_w = layer(queries, visual_kv)
            all_attn_weights.append(attn_w)

        queries = self.norm(queries)

        # Upproject to LLM embedding space
        # (batch_num, num_queries, model_dim) → (batch_num, num_queries, llm_dim)
        output = self.output_proj(queries)

        return output, all_attn_weights

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

# 4) DeepStack  *(Multi-Level Feature Fusion)*

<img src="https://deepstack-vl.github.io/figs/deepstack_teaser.png" width=500>

> **Style:** Qwen3-VL, InternVL2, LLaVA-HR  
> **When to use:** OCR, chart understanding, dense grounding — tasks requiring fine-grained spatial detail.

All three previous methods use **only the final ViT layer**.  
That's convenient but lossy: the final layer aggregates global context and discards local texture.

Different ViT layers encode different levels of visual abstraction:

```
Layer 1–3  :  Gabor-like edges, colour blobs
Layer 4–6  :  Textures, local patterns
Layer 7–9  :  Object parts, spatial relationships
Layer 10–12:  Semantic regions, global scene understanding
```

DeepStack samples features from $L$ intermediate layers, projects each to $D_l$, then **fuses** them.

In [6]:
class DeepStack(nn.Module):
    """
    Multi-level ViT feature fusion (Qwen3-VL / InternVL2 style).

    Unlike MLP / Perceiver / Q-Former which consume the final ViT layer only,
    DeepStack takes a *list* of intermediate layer outputs and fuses them.

    Two fusion modes:
        'concat'  — concatenate projections along the feature dim, then compress.
                    Most expressive; preferred when fine detail matters most.
        'sum'     — learned softmax-gated weighted sum.
                    Interpretable: inspect self.get_level_weights() to see which
                    ViT layer the model relies on most for a given task.
    """

    def __init__(
        self,
        vision_dim: int,
        llm_dim: int,
        num_levels: int = 4,
        fusion: str = "concat",  # "concat" | "sum"
    ) -> None:

        super().__init__()
        assert fusion in ("concat", "sum"), "fusion must be 'concat' or 'sum'"
        self.num_levels = num_levels
        self.fusion = fusion

        # Per-level projection + normalisation
        # Each: (batch_num, num_patches, vision_dim) → (batch_num, num_patches, llm_dim)
        self.level_projs = nn.ModuleList(
            [
                nn.Sequential(nn.Linear(vision_dim, llm_dim), nn.LayerNorm(llm_dim))
                for _ in range(num_levels)
            ]
        )

        if fusion == "concat":
            # Compress concatenated multi-level features back to llm_dim
            # (batch_num, num_patches, llm_dim * num_levels) → (batch_num, num_patches, llm_dim)
            self.fusion_proj = nn.Sequential(
                nn.Linear(llm_dim * num_levels, llm_dim * 2),
                nn.GELU(),
                nn.Linear(llm_dim * 2, llm_dim),
            )
        else:
            # Scalar gate per level — softmax ensures they sum to 1
            # (num_levels,)
            self.level_gates = nn.Parameter(torch.ones(num_levels) / num_levels)

    def forward(self, level_features: List[torch.Tensor]) -> torch.Tensor:
        """
        level_features : List[Tensor] of length num_levels
                         Each tensor: (batch_num, num_patches, vision_dim)
        Returns        : (batch_num, num_patches, llm_dim)
        """
        assert (
            len(level_features) == self.num_levels
        ), f"Expected {self.num_levels} levels, got {len(level_features)}"

        # Project each ViT level independently
        # projected[i]: (batch_num, num_patches, llm_dim)
        projected = [proj(feat) for proj, feat in zip(self.level_projs, level_features)]

        if self.fusion == "concat":
            # Concatenate along feature dimension
            # (batch_num, num_patches, llm_dim * num_levels)
            fused = torch.cat(projected, dim=-1)

            # Input: (batch_num, num_patches, llm_dim * num_levels)
            # Output: (batch_num, num_patches, llm_dim)
            return self.fusion_proj(fused)

        # Gated sum fusion
        # gates: (num_levels,) after softmax
        gates = F.softmax(self.level_gates, dim=0)

        # Weighted sum: (batch_num, num_patches, llm_dim)
        return sum(g * feat for g, feat in zip(gates, projected))

    def get_level_weights(self) -> np.ndarray:
        """Return the normalised importance of each ViT level (gated-sum only)."""
        if self.fusion == "sum":
            return F.softmax(self.level_gates, dim=0).detach().cpu().numpy()
        return np.ones(self.num_levels) / self.num_levels

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

In [7]:
# Demo
VISION_DIM = 1024
LLM_DIM = 2048
NUM_LEVELS = 4
BATCH = 2
SEQ_LEN = 32
H, W = 16, 16

model = DeepStack(VISION_DIM, LLM_DIM, NUM_LEVELS, fusion="concat")
image_feats = [torch.randn(BATCH, H * W, VISION_DIM) for _ in range(NUM_LEVELS)]
text_feats = [torch.randn(BATCH, SEQ_LEN, VISION_DIM) for _ in range(NUM_LEVELS)]

for level_feats in [image_feats, text_feats]:
    out = model(level_feats)
    if model.fusion == "sum":
        weights = model.get_level_weights()

# Summary

1. **Parameter count** — how many trainable weights does the bridge add?
2. **Output token count** — how many tokens does the LLM receive from vision?
3. **Compute budget proxy** — token count × llm_dim² (the dominant cost in LLM self-attention)

```
                HIGH DETAIL
                    │
    DeepStack ──────┤──────────────────── MLP Projector
    (multi-scale)   │                   (simple, fast)
                    │
    ────────────────┼──────────── COMPRESSION
                    │
    Q-Former ───────┤──────────── Perceiver Resampler
    (complex,       │            (elegant, moderate)
     pre-trained)   │
                    │
                LOW DETAIL
```

> **Why the field converged on MLP + MoE backbone:**
>
> - MLP projector is 10–100× simpler than Q-Former/Perceiver, trains end-to-end, and performs comparably on most benchmarks when paired with strong instruction-tuning data.
> - DeepStack is worth the overhead only when fine-grained detail is the bottleneck (OCR, charts, grounding).
> - MoE backbone provides free capacity scaling: 10× more parameters at 2–3× more compute — the most efficient way to scale a multimodal model.
